In [1]:
import os, sys, glob, json, time, zipfile, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import (
    accuracy_score, jaccard_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except Exception:
    plt.style.use('ggplot')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    torch.randn(2, 2).to(DEVICE).matmul(torch.randn(2, 2).to(DEVICE))
except Exception:
    print("CUDA not usable, switching to CPU")
    DEVICE = torch.device('cpu')

BASE_DIR = "/kaggle/input/datasets/christiancorvino/cholec80-mr-sapienza"
if not os.path.exists(BASE_DIR):
    BASE_DIR = "/kaggle/input/cholec80-mr-sapienza"
PHASE_DIR = os.path.join(BASE_DIR, "phase_annotations")
VIDEO_DIR = os.path.join(BASE_DIR, "videos")
OUTPUT_DIR = "/kaggle/working"
FEATURE_DIR = os.path.join(OUTPUT_DIR, "features")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(FEATURE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

PHASE_DICT = {
    'Preparation': 0, 'CalotTriangleDissection': 1, 'ClippingCutting': 2,
    'GallbladderDissection': 3, 'GallbladderPackaging': 4,
    'CleaningCoagulation': 5, 'GallbladderRetraction': 6
}
PHASE_NAMES = list(PHASE_DICT.keys())
NUM_CLASSES = 7

TRAIN_IDS = list(range(1, 33))
VAL_IDS = list(range(33, 41))
TEST_IDS = list(range(41, 81))

IMG_SIZE = 224
BATCH_EXTRACT = 64
NUM_STAGES = 4
NUM_LAYERS = 12
NUM_F_MAPS = 64
FEATURE_DIM = 2048
LR = 5e-4
WEIGHT_DECAY = 1e-5
EPOCHS = 150
PATIENCE = 20
SMOOTHING_LAMBDA = 0.30

print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Dataset: {BASE_DIR}")
print(f"Split — Train: {len(TRAIN_IDS)} | Val: {len(VAL_IDS)} | Test: {len(TEST_IDS)}")

Device: cuda
GPU: Tesla T4
Dataset: /kaggle/input/datasets/christiancorvino/cholec80-mr-sapienza
Split — Train: 32 | Val: 8 | Test: 40


In [2]:
def load_annotations(phase_dir, video_ids):
    annotations = {}
    for vid in video_ids:
        pf = os.path.join(phase_dir, f"video{vid:02d}-phase.txt")
        if not os.path.exists(pf):
            continue
        df = pd.read_csv(pf, sep=r'\t|,|\s+', engine='python')
        frame_idx = df.iloc[:, 0].values.astype(int)
        phases = df.iloc[:, 1].map(PHASE_DICT).values.astype(int)
        if len(frame_idx) > 1 and (frame_idx[1] - frame_idx[0]) == 1:
            mask = np.arange(0, len(frame_idx), 25)
            frame_idx = frame_idx[mask]
            phases = phases[mask]
        annotations[vid] = {'frame_indices': frame_idx, 'phases': phases}
    return annotations

all_annotations = load_annotations(PHASE_DIR, list(range(1, 81)))
print(f"Loaded annotations for {len(all_annotations)} videos")
for vid in sorted(all_annotations.keys())[:5]:
    d = all_annotations[vid]
    print(f"  Video {vid:02d}: {len(d['phases'])} frames, "
          f"indices [{d['frame_indices'][0]}–{d['frame_indices'][-1]}]")

Loaded annotations for 80 videos
  Video 01: 1734 frames, indices [0–43325]
  Video 02: 2840 frames, indices [0–70975]
  Video 03: 5829 frames, indices [0–145700]
  Video 04: 1523 frames, indices [0–38050]
  Video 05: 2345 frames, indices [0–58600]


In [3]:
needed = []
for vid in all_annotations:
    fp = os.path.join(FEATURE_DIR, f"video{vid:02d}_features.npy")
    lp = os.path.join(FEATURE_DIR, f"video{vid:02d}_labels.npy")
    if not (os.path.exists(fp) and os.path.exists(lp)):
        needed.append(vid)

print(f"Features already cached: {len(all_annotations) - len(needed)}/{len(all_annotations)}")
print(f"Videos to extract: {len(needed)}")

if needed:
    resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    resnet = nn.Sequential(*list(resnet.children())[:-1])
    resnet = resnet.to(DEVICE).eval()

    try:
        dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
        with torch.no_grad():
            resnet(dummy)
        del dummy
        torch.cuda.empty_cache()
        EXTRACT_DEVICE = DEVICE
    except Exception as e:
        print(f"CUDA test failed ({e}), falling back to CPU")
        EXTRACT_DEVICE = torch.device('cpu')
        resnet = resnet.to(EXTRACT_DEVICE)
        BATCH_EXTRACT = 16

    print(f"Feature extraction on: {EXTRACT_DEVICE}")

    img_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    for vid in tqdm(needed, desc="Extracting features"):
        ann = all_annotations[vid]
        fp = os.path.join(FEATURE_DIR, f"video{vid:02d}_features.npy")
        lp = os.path.join(FEATURE_DIR, f"video{vid:02d}_labels.npy")

        vp = os.path.join(VIDEO_DIR, f"video{vid:02d}.mp4")
        if not os.path.exists(vp):
            print(f"  Video {vid:02d} not found, skipping")
            continue

        cap = cv2.VideoCapture(vp)
        if not cap.isOpened():
            continue

        frame_indices = ann['frame_indices']
        phases = ann['phases']
        target_set = set(frame_indices.tolist())
        max_frame = int(max(target_set))
        feats_all, valid_mask, batch_buf = [], [], []
        current_frame = 0

        while current_frame <= max_frame:
            if current_frame in target_set:
                ret, frame = cap.read()
                if ret:
                    valid_mask.append(True)
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    batch_buf.append(img_transform(Image.fromarray(rgb)))
                    if len(batch_buf) >= BATCH_EXTRACT:
                        with torch.no_grad():
                            out = resnet(torch.stack(batch_buf).to(EXTRACT_DEVICE))
                        feats_all.append(out.squeeze(-1).squeeze(-1).cpu().numpy())
                        batch_buf = []
                else:
                    valid_mask.append(False)
            else:
                cap.grab()
            current_frame += 1

        if batch_buf:
            with torch.no_grad():
                out = resnet(torch.stack(batch_buf).to(EXTRACT_DEVICE))
            feats_all.append(out.squeeze(-1).squeeze(-1).cpu().numpy())
        cap.release()

        if feats_all:
            features = np.concatenate(feats_all, axis=0)
            labels = phases[np.array(valid_mask)]
            np.save(fp, features)
            np.save(lp, labels)
            print(f"  Video {vid:02d}: {features.shape[0]} frames → {features.shape}")

    del resnet
    if EXTRACT_DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print("Feature extraction complete")

Features already cached: 0/80
Videos to extract: 80
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 187MB/s]


Feature extraction on: cuda


Extracting features:   0%|          | 0/80 [00:00<?, ?it/s]

  Video 01: 1734 frames → (1734, 2048)
  Video 02: 2840 frames → (2840, 2048)
  Video 03: 5829 frames → (5829, 2048)
  Video 04: 1523 frames → (1523, 2048)
  Video 05: 2345 frames → (2345, 2048)
  Video 06: 2154 frames → (2154, 2048)
  Video 07: 4558 frames → (4558, 2048)
  Video 08: 1520 frames → (1520, 2048)
  Video 09: 2703 frames → (2703, 2048)
  Video 10: 1750 frames → (1750, 2048)
  Video 11: 3221 frames → (3221, 2048)
  Video 12: 1091 frames → (1091, 2048)
  Video 13: 982 frames → (982, 2048)
  Video 14: 1709 frames → (1709, 2048)
  Video 15: 2059 frames → (2059, 2048)
  Video 16: 2958 frames → (2958, 2048)
  Video 17: 1305 frames → (1305, 2048)
  Video 18: 1943 frames → (1943, 2048)
  Video 19: 2425 frames → (2425, 2048)
  Video 20: 1450 frames → (1450, 2048)
  Video 21: 1259 frames → (1259, 2048)
  Video 22: 1533 frames → (1533, 2048)
  Video 23: 1636 frames → (1636, 2048)
  Video 24: 1976 frames → (1976, 2048)
  Video 25: 2130 frames → (2130, 2048)
  Video 26: 1774 frames → (

In [4]:
class Cholec80FeatureDataset:
    def __init__(self, feature_dir, video_ids):
        self.data = []
        for vid in sorted(video_ids):
            fp = os.path.join(feature_dir, f"video{vid:02d}_features.npy")
            lp = os.path.join(feature_dir, f"video{vid:02d}_labels.npy")
            if not os.path.exists(fp) or not os.path.exists(lp):
                continue
            self.data.append({
                'features': torch.tensor(np.load(fp), dtype=torch.float32),
                'labels': torch.tensor(np.load(lp), dtype=torch.long),
                'video_id': vid
            })

    def __len__(self):
        return len(self.data)

train_dataset = Cholec80FeatureDataset(FEATURE_DIR, TRAIN_IDS)
val_dataset = Cholec80FeatureDataset(FEATURE_DIR, VAL_IDS)
test_dataset = Cholec80FeatureDataset(FEATURE_DIR, TEST_IDS)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)} videos")

all_train_labels = np.concatenate([d['labels'].numpy() for d in train_dataset.data])
class_counts = np.bincount(all_train_labels, minlength=NUM_CLASSES)
total = class_counts.sum()
class_weights = torch.tensor(
    total / (NUM_CLASSES * np.maximum(class_counts, 1)), dtype=torch.float32
).to(DEVICE)

print(f"\nTrain distribution ({total} frames):")
for i, name in enumerate(PHASE_NAMES):
    print(f"  {name:<30} {class_counts[i]:>7} ({class_counts[i]/total*100:5.1f}%)")
print(f"\nClass weights: {class_weights.cpu().numpy().round(3)}")

Train: 32 | Val: 8 | Test: 40 videos

Train distribution (71032 frames):
  Preparation                       3104 (  4.4%)
  CalotTriangleDissection          29730 ( 41.9%)
  ClippingCutting                   6139 (  8.6%)
  GallbladderDissection            20046 ( 28.2%)
  GallbladderPackaging              3084 (  4.3%)
  CleaningCoagulation               5979 (  8.4%)
  GallbladderRetraction             2950 (  4.2%)

Class weights: [3.269 0.341 1.653 0.506 3.29  1.697 3.44 ]


In [5]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, channels):
        super().__init__()
        self.conv_dilated = nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.conv_1x1 = nn.Conv1d(channels, channels, 1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        return x + out


class SingleStageTCN(nn.Module):
    def __init__(self, num_layers, num_f_maps, in_dim, num_classes):
        super().__init__()
        self.conv_in = nn.Conv1d(in_dim, num_f_maps, 1)
        self.layers = nn.ModuleList([
            DilatedResidualLayer(2 ** i, num_f_maps) for i in range(num_layers)
        ])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, 1)

    def forward(self, x):
        out = self.conv_in(x)
        for layer in self.layers:
            out = layer(out)
        return self.conv_out(out)


class MultiStageTCN(nn.Module):
    def __init__(self, num_stages, num_layers, num_f_maps, in_dim, num_classes):
        super().__init__()
        self.stage1 = SingleStageTCN(num_layers, num_f_maps, in_dim, num_classes)
        self.stages = nn.ModuleList([
            SingleStageTCN(num_layers, num_f_maps, num_classes, num_classes)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x):
        out = self.stage1(x)
        outputs = [out]
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1))
            outputs.append(out)
        return outputs


model = MultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, FEATURE_DIM, NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {total_params:,} total, {trainable_params:,} trainable")
print(model)

Parameters: 927,068 total, 927,068 trainable
MultiStageTCN(
  (stage1): SingleStageTCN(
    (conv_in): Conv1d(2048, 64, kernel_size=(1,), stride=(1,))
    (layers): ModuleList(
      (0): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (1): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (2): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(4,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (3): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, k

In [6]:
class TruncatedMSE(nn.Module):
    def __init__(self, tau=4.0):
        super().__init__()
        self.tau = tau

    def forward(self, logits):
        log_p = F.log_softmax(logits, dim=1)
        diff = log_p[:, :, 1:] - log_p[:, :, :-1]
        return torch.mean(torch.clamp(diff ** 2, max=self.tau ** 2))


def mstcn_loss(outputs, targets, ce_fn, smooth_fn, lam):
    loss = 0.0
    for out in outputs:
        loss += ce_fn(out, targets)
        loss += lam * smooth_fn(out)
    return loss


def train_epoch(model, dataset, optimizer, ce_fn, smooth_fn, lam):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for idx in np.random.permutation(len(dataset)):
        s = dataset.data[idx]
        x = s['features'].unsqueeze(0).permute(0, 2, 1).to(DEVICE)
        y = s['labels'].unsqueeze(0).to(DEVICE)
        optimizer.zero_grad()
        outputs = model(x)
        loss = mstcn_loss(outputs, y, ce_fn, smooth_fn, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()
        preds = outputs[-1].squeeze(0).argmax(0)
        correct += (preds == y.squeeze(0)).sum().item()
        total += y.numel()
    return total_loss / len(dataset), correct / total


@torch.no_grad()
def evaluate(model, dataset, ce_fn, smooth_fn, lam):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    per_video = {}
    for s in dataset.data:
        x = s['features'].unsqueeze(0).permute(0, 2, 1).to(DEVICE)
        y = s['labels'].unsqueeze(0).to(DEVICE)
        vid = s['video_id']
        outputs = model(x)
        total_loss += mstcn_loss(outputs, y, ce_fn, smooth_fn, lam).item()
        p = outputs[-1].squeeze(0).argmax(0).cpu().numpy()
        l = y.squeeze(0).cpu().numpy()
        all_preds.extend(p.tolist())
        all_labels.extend(l.tolist())
        per_video[vid] = {
            'preds': p, 'labels': l,
            'accuracy': accuracy_score(l, p),
            'jaccard': jaccard_score(l, p, average='macro', zero_division=0)
        }
    acc = accuracy_score(all_labels, all_preds)
    jac = jaccard_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(dataset), acc, jac, all_preds, all_labels, per_video


ce_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
smooth_criterion = TruncatedMSE(tau=4.0)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-6
)

In [7]:
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_jaccard': [], 'lr': []
}
best_val_jaccard = 0.0
best_model_state = None
patience_counter = 0
model_path = os.path.join(OUTPUT_DIR, "best_model.pt")

print(f"{'Ep':>4} {'TrLoss':>8} {'TrAcc':>7} {'VlLoss':>8} {'VlAcc':>7} {'VlJac':>7} {'LR':>10}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(
        model, train_dataset, optimizer, ce_criterion, smooth_criterion, SMOOTHING_LAMBDA
    )
    vl_loss, vl_acc, vl_jac, _, _, _ = evaluate(
        model, val_dataset, ce_criterion, smooth_criterion, SMOOTHING_LAMBDA
    )
    scheduler.step(vl_loss)
    lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['val_jaccard'].append(vl_jac)
    history['lr'].append(lr)

    star = ""
    if vl_jac > best_val_jaccard:
        best_val_jaccard = vl_jac
        best_model_state = deepcopy(model.state_dict())
        torch.save(best_model_state, model_path)
        patience_counter = 0
        star = " *"
    else:
        patience_counter += 1

    print(f"{epoch:4d} {tr_loss:8.4f} {tr_acc:7.4f} "
          f"{vl_loss:8.4f} {vl_acc:7.4f} {vl_jac:7.4f} {lr:10.2e}{star}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

print(f"\nBest val Jaccard: {best_val_jaccard:.4f}")
model.load_state_dict(best_model_state)

  Ep   TrLoss   TrAcc   VlLoss   VlAcc   VlJac         LR
------------------------------------------------------------
   1   8.1622  0.1793   7.1160  0.1418  0.1152   5.00e-04 *
   2   6.5636  0.2551   6.0506  0.2131  0.1623   5.00e-04 *
   3   5.7353  0.3937   5.2324  0.3536  0.3373   5.00e-04 *
   4   4.9407  0.4898   4.6802  0.6808  0.5858   5.00e-04 *
   5   4.4957  0.7458   4.4956  0.8337  0.7017   5.00e-04 *
   6   4.1831  0.8336   4.3267  0.8835  0.7459   5.00e-04 *
   7   3.9835  0.9049   4.0526  0.8599  0.7844   5.00e-04 *
   8   3.8339  0.9320   4.0306  0.8775  0.8039   5.00e-04 *
   9   3.7492  0.9497   4.1522  0.8399  0.7883   5.00e-04
  10   3.6701  0.9596   4.0879  0.8751  0.7906   5.00e-04
  11   3.5697  0.9714   4.2522  0.8636  0.7942   5.00e-04
  12   3.4750  0.9748   3.9943  0.8986  0.8131   5.00e-04 *
  13   3.4896  0.9713   3.8738  0.9404  0.8529   5.00e-04 *
  14   3.4431  0.9762   4.5144  0.8551  0.7177   5.00e-04
  15   3.3909  0.9779   3.9957  0.9152  0.8295   

<All keys matched successfully>

In [8]:
model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))

test_loss, test_acc, test_jac, test_preds, test_labels, test_per_video = evaluate(
    model, test_dataset, ce_criterion, smooth_criterion, SMOOTHING_LAMBDA
)

from scipy.ndimage import median_filter

for vid in test_per_video:
    test_per_video[vid]['preds'] = median_filter(test_per_video[vid]['preds'], size=21).astype(int)
    p = test_per_video[vid]['preds']
    l = test_per_video[vid]['labels']
    test_per_video[vid]['accuracy'] = accuracy_score(l, p)
    test_per_video[vid]['jaccard'] = jaccard_score(l, p, average='macro', zero_division=0)

y_true, y_pred = [], []
for vid in sorted(test_per_video.keys()):
    y_true.extend(test_per_video[vid]['labels'].tolist())
    y_pred.extend(test_per_video[vid]['preds'].tolist())
y_true = np.array(y_true)
y_pred = np.array(y_pred)

overall = {
    'accuracy': accuracy_score(y_true, y_pred),
    'jaccard_macro': jaccard_score(y_true, y_pred, average='macro', zero_division=0),
    'jaccard_weighted': jaccard_score(y_true, y_pred, average='weighted', zero_division=0),
    'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
    'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
    'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
    'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    'test_loss': test_loss,
}

per_phase = {}
for i, name in enumerate(PHASE_NAMES):
    bt = (y_true == i).astype(int)
    bp = (y_pred == i).astype(int)
    per_phase[name] = {
        'precision': precision_score(bt, bp, zero_division=0),
        'recall': recall_score(bt, bp, zero_division=0),
        'f1': f1_score(bt, bp, zero_division=0),
        'jaccard': jaccard_score(bt, bp, zero_division=0),
        'support': int(bt.sum())
    }

print("=" * 60)
print("TEST RESULTS")
print("=" * 60)
for k, v in overall.items():
    print(f"  {k:<25} {v:.4f}")
print(f"\n{'Phase':<30} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Jac':>6} {'N':>7}")
print("-" * 65)
for name, m in per_phase.items():
    print(f"{name:<30} {m['precision']:6.3f} {m['recall']:6.3f} "
          f"{m['f1']:6.3f} {m['jaccard']:6.3f} {m['support']:7d}")

TEST RESULTS
  accuracy                  0.8921
  jaccard_macro             0.7351
  jaccard_weighted          0.8105
  precision_macro           0.8371
  precision_weighted        0.8960
  recall_macro              0.8516
  recall_weighted           0.8921
  f1_macro                  0.8432
  f1_weighted               0.8925
  test_loss                 4.4568

Phase                            Prec    Rec     F1    Jac       N
-----------------------------------------------------------------
Preparation                     0.859  0.938  0.897  0.813    4816
CalotTriangleDissection         0.898  0.953  0.925  0.860   37940
ClippingCutting                 0.724  0.751  0.737  0.583    6751
GallbladderDissection           0.972  0.872  0.919  0.851   34314
GallbladderPackaging            0.744  0.777  0.760  0.613    3902
CleaningCoagulation             0.812  0.877  0.843  0.728    7113
GallbladderRetraction           0.851  0.794  0.822  0.697    3398


In [9]:
with open(os.path.join(RESULTS_DIR, "overall_metrics.txt"), 'w') as f:
    f.write("OVERALL TEST METRICS\n" + "=" * 50 + "\n\n")
    for k, v in overall.items():
        f.write(f"{k}: {v:.6f}\n")

with open(os.path.join(RESULTS_DIR, "per_phase_metrics.txt"), 'w') as f:
    f.write("PER-PHASE TEST METRICS\n" + "=" * 80 + "\n\n")
    f.write(f"{'Phase':<30} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Jaccard':>10} {'Support':>10}\n")
    f.write("-" * 80 + "\n")
    for name, m in per_phase.items():
        f.write(f"{name:<30} {m['precision']:10.4f} {m['recall']:10.4f} "
                f"{m['f1']:10.4f} {m['jaccard']:10.4f} {m['support']:10d}\n")

with open(os.path.join(RESULTS_DIR, "per_video_metrics.txt"), 'w') as f:
    f.write("PER-VIDEO TEST METRICS\n" + "=" * 50 + "\n\n")
    f.write(f"{'Video':>10} {'Accuracy':>10} {'Jaccard':>10}\n")
    f.write("-" * 32 + "\n")
    for vid in sorted(test_per_video.keys()):
        m = test_per_video[vid]
        f.write(f"{'video'+str(vid).zfill(2):>10} {m['accuracy']:10.4f} {m['jaccard']:10.4f}\n")
    accs = [v['accuracy'] for v in test_per_video.values()]
    jacs = [v['jaccard'] for v in test_per_video.values()]
    f.write("-" * 32 + "\n")
    f.write(f"{'Mean':>10} {np.mean(accs):10.4f} {np.mean(jacs):10.4f}\n")
    f.write(f"{'Std':>10} {np.std(accs):10.4f} {np.std(jacs):10.4f}\n")

with open(os.path.join(RESULTS_DIR, "training_log.txt"), 'w') as f:
    f.write(f"{'Epoch':>6} {'TrainLoss':>12} {'TrainAcc':>12} {'ValLoss':>12} "
            f"{'ValAcc':>12} {'ValJaccard':>12} {'LR':>12}\n")
    f.write("-" * 80 + "\n")
    for i in range(len(history['train_loss'])):
        f.write(f"{i+1:6d} {history['train_loss'][i]:12.6f} {history['train_acc'][i]:12.6f} "
                f"{history['val_loss'][i]:12.6f} {history['val_acc'][i]:12.6f} "
                f"{history['val_jaccard'][i]:12.6f} {history['lr'][i]:12.2e}\n")

with open(os.path.join(RESULTS_DIR, "class_distribution.txt"), 'w') as f:
    f.write("CLASS DISTRIBUTION\n" + "=" * 60 + "\n")
    for sn, sd in [("Train", train_dataset), ("Val", val_dataset), ("Test", test_dataset)]:
        labs = np.concatenate([d['labels'].numpy() for d in sd.data])
        counts = np.bincount(labs, minlength=NUM_CLASSES)
        t = counts.sum()
        f.write(f"\n{sn} ({t} frames):\n" + "-" * 50 + "\n")
        for i, name in enumerate(PHASE_NAMES):
            f.write(f"  {name:<30} {counts[i]:>8} ({counts[i]/t*100:5.1f}%)\n")

with open(os.path.join(RESULTS_DIR, "hyperparameters.txt"), 'w') as f:
    f.write("HYPERPARAMETERS\n" + "=" * 50 + "\n\n")
    f.write("Model: Multi-Stage TCN (MS-TCN)\n")
    f.write("Feature extractor: ResNet50 (ImageNet, frozen)\n\n")
    for k, v in [("NUM_STAGES", NUM_STAGES), ("NUM_LAYERS", NUM_LAYERS),
                 ("NUM_F_MAPS", NUM_F_MAPS), ("FEATURE_DIM", FEATURE_DIM),
                 ("NUM_CLASSES", NUM_CLASSES), ("IMG_SIZE", IMG_SIZE),
                 ("LR", LR), ("WEIGHT_DECAY", WEIGHT_DECAY), ("EPOCHS", EPOCHS),
                 ("PATIENCE", PATIENCE), ("SMOOTHING_LAMBDA", SMOOTHING_LAMBDA)]:
        f.write(f"{k}: {v}\n")
    f.write(f"\nTrain IDs: {TRAIN_IDS}\n")
    f.write(f"Val IDs: {VAL_IDS}\n")
    f.write(f"Test IDs: {TEST_IDS}\n")
    f.write(f"\nClass weights: {class_weights.cpu().numpy()}\n")
    f.write(f"Total params: {total_params:,}\n")
    f.write(f"Trainable params: {trainable_params:,}\n")

with open(os.path.join(RESULTS_DIR, "classification_report.txt"), 'w') as f:
    f.write("CLASSIFICATION REPORT\n" + "=" * 60 + "\n\n")
    f.write(classification_report(y_true, y_pred, target_names=PHASE_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
np.savetxt(os.path.join(RESULTS_DIR, "confusion_matrix.csv"), cm, delimiter=',',
           fmt='%d', header=','.join(PHASE_NAMES))

with open(os.path.join(RESULTS_DIR, "training_history.json"), 'w') as f:
    json.dump(history, f, indent=2)

print(f"All metrics saved to {RESULTS_DIR}")

All metrics saved to /kaggle/working/results


In [10]:
SHORT = ['Prep', 'CalotDiss', 'ClipCut', 'GallDiss', 'GallPack', 'CleanCoag', 'GallRetr']
colors7 = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=SHORT, yticklabels=SHORT, ax=axes[0])
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues', xticklabels=SHORT, yticklabels=SHORT, ax=axes[1])
axes[1].set_title('Normalized Confusion Matrix'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150, bbox_inches='tight'); plt.close()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ep = range(1, len(history['train_loss']) + 1)
axes[0,0].plot(ep, history['train_loss'], label='Train', lw=2); axes[0,0].plot(ep, history['val_loss'], label='Val', lw=2)
axes[0,0].set_title('Loss'); axes[0,0].set_xlabel('Epoch'); axes[0,0].legend(); axes[0,0].grid(True)
axes[0,1].plot(ep, history['train_acc'], label='Train', lw=2); axes[0,1].plot(ep, history['val_acc'], label='Val', lw=2)
axes[0,1].set_title('Accuracy'); axes[0,1].set_xlabel('Epoch'); axes[0,1].legend(); axes[0,1].grid(True)
axes[1,0].plot(ep, history['val_jaccard'], color='green', lw=2)
axes[1,0].set_title('Val Jaccard (Macro)'); axes[1,0].set_xlabel('Epoch'); axes[1,0].grid(True)
axes[1,1].plot(ep, history['lr'], color='red', lw=2)
axes[1,1].set_title('Learning Rate'); axes[1,1].set_xlabel('Epoch'); axes[1,1].set_yscale('log'); axes[1,1].grid(True)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "training_curves.png"), dpi=150, bbox_inches='tight'); plt.close()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
f1s = [per_phase[n]['f1'] for n in PHASE_NAMES]
jacs_ph = [per_phase[n]['jaccard'] for n in PHASE_NAMES]
b1 = axes[0].bar(SHORT, f1s, color=colors7, edgecolor='black', lw=0.5)
axes[0].set_title('F1 per Phase'); axes[0].set_ylim(0, 1.1)
for bar, v in zip(b1, f1s): axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{v:.3f}', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=45)
b2 = axes[1].bar(SHORT, jacs_ph, color=colors7, edgecolor='black', lw=0.5)
axes[1].set_title('Jaccard per Phase'); axes[1].set_ylim(0, 1.1)
for bar, v in zip(b2, jacs_ph): axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{v:.3f}', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "per_phase_bar.png"), dpi=150, bbox_inches='tight'); plt.close()

ex_vids = sorted(test_per_video.keys())[:3]
fig, axes = plt.subplots(len(ex_vids), 1, figsize=(20, 4 * len(ex_vids)))
if len(ex_vids) == 1: axes = [axes]
pc = plt.cm.Set1(np.linspace(0, 1, NUM_CLASSES))
for ax, vid in zip(axes, ex_vids):
    p, l = test_per_video[vid]['preds'], test_per_video[vid]['labels']
    t = np.arange(len(l))
    for pid in range(NUM_CLASSES):
        ax.fill_between(t, 7.5, 8.5, where=(l==pid), color=pc[pid], alpha=0.8)
        ax.fill_between(t, 6.0, 7.0, where=(p==pid), color=pc[pid], alpha=0.8)
    ax.set_yticks([6.5, 8.0]); ax.set_yticklabels(['Pred', 'GT']); ax.set_ylim(5.5, 9.0)
    ax.set_xlabel('Frame (1fps)')
    ax.set_title(f'Video {vid:02d} — Acc:{test_per_video[vid]["accuracy"]:.3f} | Jac:{test_per_video[vid]["jaccard"]:.3f}')
patches = [plt.Rectangle((0,0),1,1, fc=pc[i], alpha=0.8) for i in range(NUM_CLASSES)]
fig.legend(patches, SHORT, loc='lower center', ncol=NUM_CLASSES, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "phase_timeline.png"), dpi=150, bbox_inches='tight'); plt.close()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (sn, sd) in zip(axes, [("Train", train_dataset), ("Val", val_dataset), ("Test", test_dataset)]):
    labs = np.concatenate([d['labels'].numpy() for d in sd.data])
    counts = np.bincount(labs, minlength=NUM_CLASSES)
    ax.bar(SHORT, counts, color=colors7, edgecolor='black', lw=0.5)
    ax.set_title(f'{sn} Distribution'); ax.set_ylabel('Frames'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "class_distribution.png"), dpi=150, bbox_inches='tight'); plt.close()

fig, ax = plt.subplots(figsize=(14, 5))
vids = sorted(test_per_video.keys())
va = [test_per_video[v]['accuracy'] for v in vids]
vj = [test_per_video[v]['jaccard'] for v in vids]
x = np.arange(len(vids))
ax.bar(x-0.175, va, 0.35, label='Accuracy', color='steelblue', alpha=0.8)
ax.bar(x+0.175, vj, 0.35, label='Jaccard', color='coral', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([f'V{v:02d}' for v in vids], rotation=90, fontsize=7)
ax.set_title('Per-Video Test Metrics'); ax.set_ylabel('Score'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "per_video_bar.png"), dpi=150, bbox_inches='tight'); plt.close()

print("All plots saved")

All plots saved


In [11]:
zip_path = os.path.join(OUTPUT_DIR, "cholec80_output.zip")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(RESULTS_DIR):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.join("results", f))
    mp = os.path.join(OUTPUT_DIR, "best_model.pt")
    if os.path.exists(mp):
        zf.write(mp, "best_model.pt")

print(f"Zip created: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
with zipfile.ZipFile(zip_path, 'r') as zf:
    print("\nContents:")
    for info in zf.infolist():
        print(f"  {info.filename} ({info.file_size/1024:.1f} KB)")

Zip created: /kaggle/working/cholec80_output.zip (3.9 MB)

Contents:
  results/classification_report.txt (0.8 KB)
  results/training_log.txt (3.3 KB)
  results/confusion_matrix.csv (0.3 KB)
  results/overall_metrics.txt (0.3 KB)
  results/per_phase_bar.png (73.1 KB)
  results/per_phase_metrics.txt (0.9 KB)
  results/per_video_bar.png (33.1 KB)
  results/hyperparameters.txt (0.8 KB)
  results/class_distribution.txt (1.3 KB)
  results/confusion_matrix.png (142.1 KB)
  results/training_curves.png (170.8 KB)
  results/phase_timeline.png (96.9 KB)
  results/training_history.json (5.0 KB)
  results/per_video_metrics.txt (1.5 KB)
  results/class_distribution.png (83.7 KB)
  best_model.pt (3697.6 KB)
